# Operon Visualization

Driver notebook for per-operon visual inspection of the **459** Syn1 operons
(`operons.candidate_blocks.tsv`). The plotting function `plot_one_operon` is defined
in `Operon_Visualization.py` (imported as `ov`); this notebook only drives it.

**Run from `Syn1_Operon/`** in an environment with `biopython`, `pandas`, and
`matplotlib` (e.g. conda env `RNAseq`).

**Outputs** (`operon_plots/`, two variants per operon — the names the
Genome_Reduction pipeline and `Operon_Annotation.py` consume):
- `operon_plots/<operon_id>.pdf` — genes + PacBio isoforms
- `operon_plots/<operon_id>_wdepth.pdf` — the same plus the operon-strand depth track

Stale plots from a previous segmentation are cleared first, so the folder ends up
holding exactly the current operon set. Showcase regions (ATP synthase, DCW) are
plotted at the end from manual coordinates.

In [1]:
import os, glob, time
import pandas as pd
import numpy as np
import importlib
import Operon_Visualization as ov   # plot_one_operon is defined here
importlib.reload(ov)                # pick up edits to the module without restarting

os.makedirs('operon_plots', exist_ok=True)

op = pd.read_csv('operons.candidate_blocks.tsv', sep='\t')
print(f'Total operons: {len(op)}')
print(op['segmentation_type'].value_counts().to_string())

[Operon_Visualization] genes 911 (syn3A-named 365) | isoforms 267258 | deletions 95


[Operon_Visualization] genes 911 (syn3A-named 365) | isoforms 267258 | deletions 95
Total operons: 459
segmentation_type
isoform_operon           237
rescue_single            162
isoform_operon_merged     37
rescue_multiple           22
isoform_gene_combined      1


## Clear stale plots

Remove any `operon_plots/*.pdf` left from a previous segmentation so the folder ends
up with exactly the current operon IDs (no orphaned `OP_004xx` plots from the old
480-operon map).

In [2]:
stale = glob.glob('operon_plots/*.pdf')
for f in stale:
    os.remove(f)
print(f'Removed {len(stale)} stale plot(s) from operon_plots/')

Removed 918 stale plot(s) from operon_plots/


## Plot all operons

For every operon, render two PDFs: a gene + isoform panel (`<id>.pdf`) and the same
with the operon-strand PacBio depth track (`<id>_wdepth.pdf`).

In [3]:
t0 = time.time()
n = len(op)
for i, (_, operon) in enumerate(op.iterrows(), 1):
    opid = operon['operon_id']
    ov.plot_one_operon(operon, save_path=f'operon_plots/{opid}.pdf',
                       dpi=300, PLOT_DEPTH=False)
    ov.plot_one_operon(operon, save_path=f'operon_plots/{opid}_wdepth.pdf',
                       dpi=300, PLOT_DEPTH=True)
    if i % 50 == 0 or i == n:
        print(f'  {i}/{n} operons plotted ({time.time()-t0:.0f}s)')
print(f'Done: {n} operons x 2 variants -> {2*n} PDFs in operon_plots/')

  50/459 operons plotted (55s)


  100/459 operons plotted (108s)


  150/459 operons plotted (161s)


  200/459 operons plotted (213s)


  250/459 operons plotted (266s)


  300/459 operons plotted (316s)


  350/459 operons plotted (367s)


  400/459 operons plotted (419s)


  450/459 operons plotted (469s)


  459/459 operons plotted (478s)
Done: 459 operons x 2 variants -> 918 PDFs in operon_plots/


## Example / showcase operons

Specific regions plotted from manual coordinates (isoform read threshold 50), used as
figure showcases rather than part of the per-operon QC set above.

In [4]:
# ATP synthase operon (manual coordinates), isoform read threshold 50
atpsynthase_OP = pd.DataFrame([{
    "chrom": "CP002027.1",
    "strand": "-",
    "start0": 929590 - 200,
    "end0": 936313 + 200,
    "operon_id": "ATP_Synthase",
    "sense_gene_loci": "MMSYN1_0789,MMSYN1_0790,MMSYN1_0791,MMSYN1_0792,MMSYN1_0793,MMSYN1_0794,MMSYN1_0795,MMSYN1_0796,MMSYN1_0797",
}])
ov.plot_one_operon(atpsynthase_OP.iloc[0], save_path='./ATP_Synthase_wdepth.pdf', dpi=300, PLOT_DEPTH=True, isoform_reads_threshold=50)

In [5]:
# DCW (division / cell-wall) region (manual coordinates), isoform read threshold 50
DCW_Region = pd.DataFrame([{
    "chrom": "CP002027.1",
    "strand": "-",
    "start0": 623616 - 500,
    "end0": 628640 + 50,
    "operon_id": "DCW",
    "sense_gene_loci": "MMSYN1_0521,MMSYN1_0522,MMSYN1_0523,MMSYN1_0524,MMSYN1_0525,MMSYN1_0526,MMSYN1_0527",
}])
ov.plot_one_operon(DCW_Region.iloc[0], save_path='./DCW_wdepth.pdf', dpi=300, PLOT_DEPTH=True, isoform_reads_threshold=50)